In [1]:
import numpy as np
from PIL import Image
import torch
import pandas as pd
#import torch_directml
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
#import cv2
import albumentations as A
import torchvision
from torchvision import transforms
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import os
from torch.optim import AdamW 
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.patches as patches
from torchvision.models.detection.rpn import AnchorGenerator

c:\Users\srene\Computer Vision\RIVA-challenge\RIVA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First me must specify the paths to all files needed for training and testing of the neural network.

In [2]:
CSV_PATH_TRAIN = './RIVA/annotations/annotations/train.csv'
CSV_PATH_VAL = './RIVA/annotations/annotations/val.csv'
TRAIN_PATH = './RIVA/images/images/train'
VAL_PATH = './RIVA/images/images/val'
TEST_PATH = './RIVA/images/images/test'

We will be using DirectML because all training and testing will be done on an AMD RX 5700xt

In [3]:
device = torch_directml.device()

For the transformations applied initially I decided to use simple ones. The choice of the library Albumentations instead of Pytorch for them is because the latter doesn't support bounding boxes and it would imply coding them.

In [4]:
transform_train = A.Compose([
    A.Resize(512,512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    
    # CORRECCION 1: Usamos Affine en lugar de ShiftScaleRotate
        # - scale: rango absoluto (antes era relativo +/- 0.15)
        # - translate_percent: equivalente a shift_limit
        # - cval: reemplaza a 'value' para el relleno de bordes
    A.Affine(
            scale=(0.85, 1.15),
            translate_percent={'x': (-0.0625, 0.0625), 'y': (-0.0625, 0.0625)},
            rotate=(-45, 45),
            p=0.5
    ),

    # --- 2. Transformaciones de Color y Ruido (Simulación de Dominio) ---
    
    # Simula diferencias en la "tinción" (Staining) y la fuente de luz.
    # En patología, el color exacto importa menos que el contraste cromatina/citoplasma.
    A.OneOf([
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=1),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
    ], p=0.7),
    
    # Simula problemas de foco en el microscopio o desenfoque por movimiento.
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1),
        A.MotionBlur(blur_limit=5, p=1),
    ], p=0.3),
    
    # Ruido de sensor (común en digitalización de slides).
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.3),

    # --- 3. Normalización y Formato ---
    
    # Usamos las medias y stds de ImageNet porque tu backbone es ResNet50 preentrenado.
    # Aunque el dominio es distinto, el backbone espera esta distribución inicial.
    A.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225], 
        max_pixel_value=255.0, 
        p=1.0
    ),
    
    # Convertir a Tensor de PyTorch (C, H, W)
    A.ToTensorV2()
  ], telemetry=False, bbox_params=A.BboxParams(format='pascal_voc', # Specifying the input format
                           label_fields=['labels'], # Specifying the label name
                           min_visibility=0.3     # Box must be 30% visible after crop
                           ))
transform_test = A.Compose([
    A.Resize(512, 512),
    A.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225], 
            max_pixel_value=255.0, 
            p=1.0
        ),
    A.ToTensorV2()
], telemetry=False, bbox_params=A.BboxParams(format='pascal_voc', 
                           label_fields=['labels'] 
                           ))

We will have to build a new Dataset subclass that I chose to name BethesdaDataset because of the name of the classifications.

The required minimum protocol for a Dataset type in Pytorch is __len__ and __getitem__, naturally BethesdaDataset follows it

In [4]:
class BethesdaDataset(Dataset):
    def __init__(self, csv_file, root_dir, transforms=None):
        self.root_dir = root_dir
        self.transforms = transforms

        # Loading the CSV
        self.df = pd.read_csv(csv_file)

        # Collecting the unique image_ids
        self.image_ids = self.df['image_filename'].unique()

    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx] # id is reserved by python

        records = self.df[self.df['image_filename'] == image_id]

        image_path = os.path.join(self.root_dir, image_id) # It returns e.g. /images/image_01
        image = np.array(Image.open(image_path).convert('RGB')) # Converted to np array so Albumentations library supports it

        H, W, _ = image.shape

        boxes = []
        labels = []

        for _, row in records.iterrows():
            #print(f"Imagen: {image_id} | Tamaño: {W}x{H} | Caja X: {row['x']}")
            x_center, y_center = row['x'] , row['y']
            width, height = row['width'], row['height']
            class_id = row['class'] + 1

            x_min = x_center - (width / 2)
            y_min = y_center - (height / 2)
            x_max = x_center + (width / 2)
            y_max = y_center + (height / 2)

            x_min = max(0, min(x_min, W))
            y_min = max(0, min(y_min, H))
            x_max = max(0, min(x_max, W))
            y_max = max(0, min(y_max, H))

            if (x_max <= x_min) or (y_max <= y_min):
                continue

            # We have to adapt the center based description to the one used in FasterRCNN
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(class_id) # 0 is reserved for the background

        if self.transforms is not None:
            transformed = self.transforms(image=image, bboxes=boxes, labels=labels)

            image = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']

        if len(boxes) > 0:
            # Si hay cajas, convertimos a tensor normal
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            
            # Safety check: asegúrate que sea (N, 4) y no (N, 4, 1) o algo raro
            if boxes.ndim == 1 and len(boxes) == 4:
                boxes = boxes.unsqueeze(0) # Caso de una sola caja plana
                 
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
            # Calcular area e iscrowd (requerido por COCO eval)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
            
        else:
            # CASO CERO CAJAS: Esto es lo que rompe tu training loop
            # Debemos crear un tensor vacío pero con la DIMENSIÓN CORRECTA (0, 4)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        """
        # Converting the lists to tensors
        boxes = torch.as_tensor(boxes, dtype = torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64) + 1

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) # (x_max - x_min) * (y_max - y_min)

        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        """
        tensor_image_id = torch.as_tensor([idx])

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = tensor_image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        return image, target


By default pytorch stacks images but in this case we can't do that because we can and will have different number of bounding boxes per image.

In [6]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [10]:
train_ds = BethesdaDataset(csv_file=CSV_PATH_TRAIN, root_dir=TRAIN_PATH, transforms=transform_train)
test_ds = BethesdaDataset(csv_file=CSV_PATH_VAL, root_dir=VAL_PATH, transforms=transform_test)

NameError: name 'transform_train' is not defined

In [8]:
len(train_ds)

828

In [9]:
train_loader = DataLoader(dataset=train_ds, shuffle=True, batch_size=2, collate_fn=collate_fn)
test_loader = DataLoader(dataset=test_ds, shuffle=False, batch_size=2, collate_fn=collate_fn)

In [10]:
def get_mean_and_std(loader):
    channels_sum = torch.zeros(3).float()
    channels_squared_sum = torch.zeros(3).float()
    total_pixels = 0

    for data, _ in tqdm(loader):
        for image in data:

            image = image.double() / 255 #Divide by 255 to have values between [0,1]

            channels_sum += torch.sum(image, dim=[1, 2]) # dim =[0,2,3]
            channels_squared_sum += torch.sum(image**2, dim=[1, 2])
            
            num_pixels = image.shape[1] * image.shape[2] # Height * Width, excluding image.shape[1] (the channel number)
            total_pixels += num_pixels

    mean = channels_sum / total_pixels

    variance = (channels_squared_sum / total_pixels) - (mean ** 2)
    
    std = torch.sqrt(torch.as_tensor(variance))
    
    return mean, std

In [11]:
#print(get_mean_and_std(train_loader))

In [12]:
def show_image(image):
    plt.figure(figsize=(5.12, 5.12))
    plt.imshow(image.permute(1,2,0))
    plt.show()

def show_image_and_bounding(image, labels):
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = image_np.astype(np.uint8)

    image_cv = np.ascontiguousarray(image_np)
    
    image_cv = cv2.cvtColor(image_cv, cv2.COLOR_RGB2BGR)

    boxes = labels[1]["boxes"].tolist()
    for box in boxes:
        start_point = (int(box[0]), int(box[1]))
        print(start_point)
        end_point = (int(box[2]), int(box[3]))
        cv2.rectangle(image_cv, start_point, end_point, color=(0,255,0), thickness=1)

    cv2.imwrite("example_with_bounding_boxes.jpg", image_cv)

    plt.figure(figsize=(5.12, 5.12))
    plt.imshow(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
"""model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    trainable_backbone_layers = 3
)


anchor_sizes = ((64,), (80,), (96,), (112,), (128,))  # Centrado en 100x100
aspect_ratios = ((1.0,),) * len(anchor_sizes)  # Solo cuadrados (aspect ratio 1:1)

anchor_generator = AnchorGenerator(
    sizes=anchor_sizes,
    aspect_ratios=aspect_ratios
)

# Reemplazar el anchor generator del modelo
model.rpn.anchor_generator = anchor_generator

num_anchors = anchor_generator.num_anchors_per_location()[0]
rpn_head_out_channels = model.rpn.head.conv[0][0].out_channels

model.rpn.head.cls_logits = torch.nn.Conv2d(
    rpn_head_out_channels,
    num_anchors,
    kernel_size=1,
    stride=1
)

model.rpn.head.bbox_pred = torch.nn.Conv2d(
    rpn_head_out_channels,
    num_anchors * 4,
    kernel_size=1,
    stride=1
)


# We have to change the final head for the RIVA dataset that has 9 classes: background and the other 8 Bethesda categories
classes_number = 10
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Doble verificación de seguridad:
# Recorremos el modelo y forzamos a eval() cualquier capa de BN
for name, module in model.backbone.named_modules():
    if "batch_norm" in name or isinstance(module, torch.nn.BatchNorm2d):
        module.eval() # Usa media/var guardada, ignora el batch actual
        module.weight.requires_grad = False
        module.bias.requires_grad = False

# 3. ESTRATEGIA CONTRA CATASTROPHIC FORGETTING (Nivel Arquitectura)
    # Congelamos las primeras capas convolucionales (conv1 y layer1).
    # Estas capas contienen filtros de Gabor y detectores de bordes básicos 
    # que son invariantes al dominio.
for name, parameter in model.backbone.body.named_parameters():
        # 'layer1' y anteriores se congelan. 
        # 'layer2', 'layer3', 'layer4' y FPN se mantienen entrenables (con cuidado).
        if 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
            parameter.requires_grad = False


model.roi_heads.box_predictor = FastRCNNPredictor(in_features, classes_number)"""

from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from torchvision.models.detection.rpn import AnchorGenerator
# 1. Cargar el Backbone base (Classification)
    # Usamos los pesos de ImageNet-1K
backbone_base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# 2. Identificar las capas de salida para el FPN
# ConvNeXt tiene 4 stages. En la implementación de torchvision, la estructura es:
# features.0 -> Stem
# features.1 -> Stage 1 
# features.2 -> Downsample
# features.3 -> Stage 2
# ... y así sucesivamente.
# Queremos las salidas de los stages 1, 2, 3 y 4.

# Mapeo: Nombre interno -> Nombre para el FPN ('0', '1', '2', '3')
return_layers = {
    'features.1': '0',  # Stride 4
    'features.3': '1',  # Stride 8
    'features.5': '2',  # Stride 16
    'features.7': '3'   # Stride 32
}

# 3. Definir los canales de entrada de cada stage (ConvNeXt Tiny specs)
# Estos números son fijos para la arquitectura Tiny.
in_channels_list = [96, 192, 384, 768]

# 4. Crear el Feature Extractor
# Esto corta el modelo y nos devuelve un dict con los tensores de esos stages.
backbone_extractor = create_feature_extractor(backbone_base, return_layers)

# 5. Envolver con FPN (Feature Pyramid Network)
# out_channels=256 es el estándar para la salida del FPN hacia el RPN/ROI.
backbone = BackboneWithFPN(
    backbone_extractor, 
    return_layers=return_layers,
    in_channels_list=in_channels_list,
    out_channels=256,
    extra_blocks=None # No necesitamos bloques extra (Pool) para Faster R-CNN, solo para RetinaNet
)

# 6. Definir el Anchor Generator
# Como FPN devuelve mapas en 4 escalas (y un pool extra a veces), definimos tamaños.
# Tamaños base para células (ajustable si tus células son muy chicas/grandes)
anchor_sizes = ((32,), (64,), (128,), (256,), (512,)) 
aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)

rpn_anchor_generator = AnchorGenerator(
    sizes=anchor_sizes,
    aspect_ratios=aspect_ratios
)

# 7. Ensamblar el modelo final
model = FasterRCNN(
    backbone,
    num_classes=10,
    rpn_anchor_generator=rpn_anchor_generator,
    # Opcional: Ajustar box_roi_pool si quisieras, el default suele estar bien (7x7)
)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\srene/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:09<00:00, 11.7MB/s] 


ValueError: return_layers are not present in model

In [ ]:
def get_model_params_groups(model):
    """
    Separa los parámetros en dos grupos para aplicar Differential Learning Rates.
    Grupo 1: Backbone (ResNet+FPN) -> Necesita un LR bajo para no romper pesos.
    Grupo 2: ROI Heads (RPN + BoxPredictor) -> Necesita un LR alto para aprender rápido.
    """
    backbone_params = []
    head_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue # Ignoramos lo congelado
            
        if "backbone" in name:
            backbone_params.append(param)
        else:
            # RPN y ROI Heads
            head_params.append(param)
            
    return backbone_params, head_params

In [ ]:
accumulation_steps = 8

In [ ]:
def configure_optimization_differential(model, train_dataloader, epochs):
    # Hiperparámetros
    HEAD_LR = 1e-3       # Velocidad normal para lo nuevo
    BACKBONE_LR = 1e-4   # 10x más lento para lo pre-entrenado (Safety margin)
    MIN_LR = 1e-6
    WEIGHT_DECAY = 1e-5
    WARMUP_PERCENTAGE = 0.05

    # 1. Obtener grupos de parámetros
    backbone_params, head_params = get_model_params_groups(model)

    # 2. Definir AdamW con grupos de parámetros
    optimizer = AdamW([
        {
            'params': head_params, 
            'lr': HEAD_LR
        },
        {
            'params': backbone_params, 
            'lr': BACKBONE_LR
        }
    ], weight_decay=WEIGHT_DECAY)

    # 3. Calcular Steps (Igual que antes)
    steps_per_epoch = len(train_dataloader) // accumulation_steps
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_PERCENTAGE)
    cosine_steps = total_steps - warmup_steps

    # 4. Schedulers
    # Nota: Los schedulers de PyTorch manejan los grupos de LR automáticamente,
    # escalando cada grupo proporcionalmente a su LR inicial.
    
    scheduler_warmup = LinearLR(
        optimizer, start_factor=0.001, end_factor=1.0, total_iters=warmup_steps
    )
    
    scheduler_cosine = CosineAnnealingLR(
        optimizer, T_max=cosine_steps, eta_min=MIN_LR
    )

    scheduler = SequentialLR(
        optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_steps]
    )

    return optimizer, scheduler

In [ ]:
def configure_optimization_convnext(model, train_dataloader, epochs, accumulation_steps=8):
    # --- HIPERPARÁMETROS ESPECÍFICOS PARA CONVNEXT ---
    HEAD_LR = 1e-3       # Rápido para aprender las clases de RIVA
    BACKBONE_LR = 1e-4   # Lento para preservar features preentrenados
    MIN_LR = 1e-6
    
    # CRÍTICO: Weight Decay alto para pesos, CERO para biases/norm
    WEIGHT_DECAY = 0.05  # Valor estándar del paper de ConvNeXt
    
    # 1. Separación Inteligente de Parámetros
    # Necesitamos 4 grupos:
    # A. Head (Weights) -> WD: 0.05, LR: High
    # B. Head (Bias/Norm) -> WD: 0.0,  LR: High
    # C. Backbone (Weights) -> WD: 0.05, LR: Low
    # D. Backbone (Bias/Norm) -> WD: 0.0,  LR: Low
    
    groups = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
            
        # Determinar si es backbone o head
        is_backbone = "backbone" in name
        lr = BACKBONE_LR if is_backbone else HEAD_LR
        
        # Determinar si aplicamos Weight Decay
        # Regla: Si es bias o es LayerNorm (1D tensor), NO aplicar decay.
        if param.ndim <= 1 or name.endswith(".bias") or "norm" in name:
            wd = 0.0
        else:
            wd = WEIGHT_DECAY
            
        groups.append({
            "params": [param],
            "lr": lr,
            "weight_decay": wd
        })

    # 2. Optimizador AdamW
    # bettas=(0.9, 0.999) es default, pero ConvNeXt suele usar (0.9, 0.95) en ImageNet.
    # Para Fine-Tuning en RIVA, (0.9, 0.999) es más seguro.
    optimizer = AdamW(groups) # No pasamos args globales, todo está en los grupos

    # 3. Scheduler (Igual que antes, ajustado por acumulación)
    steps_per_epoch = len(train_dataloader) // accumulation_steps
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * 0.05) # 5% Warmup
    cosine_steps = total_steps - warmup_steps
    
    print(f"ConvNeXt Optimizer Configured: Total Steps={total_steps}, WD={WEIGHT_DECAY}")

    scheduler_warmup = LinearLR(optimizer, start_factor=0.001, end_factor=1.0, total_iters=warmup_steps)
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=cosine_steps, eta_min=MIN_LR)
    
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_steps])

    return optimizer, scheduler

In [ ]:
num_epochs = 100
optimizer, lr_scheduler = configure_optimization_convnext(model, train_loader, num_epochs)
"""
num_batches_per_epoch = len(train_loader)
num_steps = num_batches_per_epoch * num_epochs

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, step_size=3, gamma=0.1)
"""

'\nnum_batches_per_epoch = len(train_loader)\nnum_steps = num_batches_per_epoch * num_epochs\n\nlr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, step_size=3, gamma=0.1)\n'

In [ ]:
# Saving the original function
_original_bce_with_logits = torch.nn.functional.binary_cross_entropy_with_logits

# Log Sigmoid (Sigmoid + BCE) is not supported in DirectML but they are on their own
def patched_bce_with_logits(input, target, weight=None, size_average=None, reduce=None, reduction='mean', pos_weight=None):
    """
    Manual replacement: Divides the composite function in Sigmoid + BCELoss.
    """
    # Si se usa pos_weight, volvemos a la original (lenta) porque la matemática es compleja
    if pos_weight is not None:
        return _original_bce_with_logits(input, target, weight, size_average, reduce, reduction, pos_weight)
    
    return torch.nn.functional.binary_cross_entropy(
        torch.sigmoid(input), 
        target, 
        weight=weight, 
        size_average=size_average, 
        reduce=reduce, 
        reduction=reduction
    )

# We replace the original function for the new one so that when FastRCNN calls it, it uses the new one
torch.nn.functional.binary_cross_entropy_with_logits = patched_bce_with_logits

In [ ]:
def visualize_predictions(image, targets, predictions, score_threshold=0.3):
    """
    Visualiza ground truth (verde) vs predicciones (rojo)
    """
    # Convertir imagen a numpy
    img_np = image.cpu().permute(1, 2, 0).numpy()
    
    # Desnormalizar
    mean = [0.7490, 0.7711, 0.8236]
    std = [0.2226, 0.2171, 0.1551]
    img_np = img_np * std + mean
    img_np = np.clip(img_np, 0, 1)
    
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(img_np)
    
    # Ground truth boxes (VERDE)
    gt_boxes = targets['boxes'].cpu().numpy()
    gt_labels = targets['labels'].cpu().numpy()
    
    for box, label in zip(gt_boxes, gt_labels):
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor='green', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'GT:{label}', color='green', fontsize=8, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # Predicciones (ROJO)
    pred_boxes = predictions['boxes'].cpu().numpy()
    pred_labels = predictions['labels'].cpu().numpy()
    pred_scores = predictions['scores'].cpu().numpy()
    
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        if score < score_threshold:
            continue
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)
        ax.text(x1, y2+15, f'Pred:{label} ({score:.2f})', color='red', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    plt.title(f'Green=GT | Red=Predictions | GT boxes: {len(gt_boxes)} | Pred boxes: {len(pred_boxes)}')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'prediction_vs_gt_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:

import math
import sys

def train_one_epoch(model, optimizer, scheduler, data_loader, device, epoch, print_freq=50):
    model.train()
    
    # Trackers para el promedio de las pérdidas
    running_loss = 0.0
    running_loss_classifier = 0.0
    running_loss_box_reg = 0.0
    running_loss_objectness = 0.0
    running_loss_rpn_box_reg = 0.0
    
    # Barra de progreso para visualización profesional
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch}", unit="batch")

    for batch_idx, (images, targets) in enumerate(progress_bar):
        # 1. Mover datos a la GPU
        # Nota: 'images' es una lista de tensores, 'targets' es una lista de diccionarios.
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # 2. Forward Pass
        # Faster R-CNN devuelve un dict con las 4 losses cuando está en train()
        loss_dict = model(images, targets)

        # 3. Calcular Loss Total
        losses = sum(loss for loss in loss_dict.values())
        loss_value = losses.item()

        # Safety Check: Si la loss explota a infinito o NaN, paramos todo.
        if not math.isfinite(loss_value):
            print(f"Loss is {loss_value}, stopping training")
            print(loss_dict)
            sys.exit(1)

        loss_scaled = losses / accumulation_steps
        loss_scaled.backward()

        # --- 3. Step Condicional ---
        if (batch_idx + 1) % accumulation_steps == 0:
            # Solo actualizamos pesos cada 'accumulation_steps' iters
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            
            optimizer.step()
            scheduler.step()
            
            # Limpiamos gradientes DESPUES del step
            optimizer.zero_grad()

        # 5. Logging y Monitoreo
        # Actualizamos los acumuladores para reportar
        running_loss += loss_value
        running_loss_classifier += loss_dict['loss_classifier'].item()
        running_loss_box_reg += loss_dict['loss_box_reg'].item()
        running_loss_objectness += loss_dict['loss_objectness'].item()
        running_loss_rpn_box_reg += loss_dict['loss_rpn_box_reg'].item()
        
        # Actualizar barra de progreso con métricas actuales
        current_lr = optimizer.param_groups[0]["lr"]
        progress_bar.set_postfix({
            "Loss": f"{loss_value:.4f}",
            "Cls": f"{loss_dict['loss_classifier'].item():.3f}",
            "Box": f"{loss_dict['loss_box_reg'].item():.3f}",
            "LR": f"{current_lr:.6f}"
        })

    # Resumen al final de la época
    epoch_loss = running_loss / len(data_loader)
    print(f"\n--- End of Epoch {epoch} Summary ---")
    print(f"Total Loss: {epoch_loss:.4f}")
    print(f" > Classifier Loss: {running_loss_classifier/len(data_loader):.4f}")
    print(f" > Box Reg Loss:    {running_loss_box_reg/len(data_loader):.4f}")
    print(f" > RPN Objectness:  {running_loss_objectness/len(data_loader):.4f}")
    print(f" > RPN Box Reg:     {running_loss_rpn_box_reg/len(data_loader):.4f}")
    
    return epoch_loss

# Función maestra para correr todo el proceso
def run_training(model, train_loader, epochs, device):
    # Configurar optimizador con la estrategia definida anteriormente
    optimizer, scheduler = configure_optimization_differential(model, train_loader, epochs)
    
    model.to(device)
    
    print(f"Iniciando entrenamiento en: {device}")
    
    metric = MeanAveragePrecision()
    metric.to(device)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, optimizer, scheduler, train_loader, device, epoch)

        model.eval()

        with torch.no_grad(): # Por cada epoch pruebo el accuracy del modelo con el dataset de testing
            for test_inputs, test_targets in tqdm(test_loader, desc='Calculating mAP'):
                test_inputs = list(image.to(device) for image in test_inputs) 
                test_outputs = model(test_inputs)
            
            test_outputs_cpu = []
            for out in test_outputs:
                # Movemos cada tensor del diccionario a cpu
                test_outputs_cpu.append({k: v.cpu() for k, v in out.items()})
                
            test_targets_cpu = []
            for t in test_targets:
                # Los targets originales podrían estar en GPU o CPU, aseguramos CPU
                test_targets_cpu.append({k: v.cpu() for k, v in t.items()})
            
                # 3. Actualizar métrica con datos en RAM (CPU)
            #show_image_and_bounding(test_inputs[0], test_targets[0])
            metric.update(test_outputs_cpu, test_targets_cpu)

            result = metric.compute()
            print(f"Validation mAP (IoU=0.50:0.95): {result['map_50']:.4f}")

            if result['map_50'] > 0.8: 
                torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")
        
        # Guardado de Checkpoint
        # Guardamos el estado del modelo, optimizer y scheduler para poder reanudar
        """torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': train_loss,
        }, f"riva_fasterrcnn_epoch_{epoch}.pth")"""
        
    print("Entrenamiento finalizado.")

run_training(model, train_loader, 100, device)

Iniciando entrenamiento en: privateuseone:0


Epoch 1:   0%|          | 0/414 [00:00<?, ?batch/s]c:\Users\srene\miniconda3\envs\pytdml\lib\site-packages\torch\nn\functional.py:3172: UserWarning: The operator 'aten::binary_cross_entropy' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch._C._nn.binary_cross_entropy(input, target, weight, reduction_enum)
Epoch 1:  87%|████████▋ | 362/414 [04:56<00:42,  1.22batch/s, Loss=3.2120, Cls=1.738, Box=0.023, LR=0.000177]


RuntimeError: unknown error

In [ ]:
metric = MeanAveragePrecision()

model.to(device)

for epoch in range(num_epochs):
    
  epoch_loss = 0

  model.train(mode=True)
  accumulation_steps = 32 # making an effective batch size of 64
  optimizer.zero_grad()

  print(f"Epoch {epoch+1}/{num_epochs}")
  for i, (inputs, targets) in enumerate(tqdm(train_loader)):

    inputs = list(image.to(device) for image in inputs) 
    targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

    loss_dict = model(inputs, targets)
    losses = sum(loss for loss in loss_dict.values())


    (losses / accumulation_steps).backward()
    if (i+1) % accumulation_steps == 0: # Implementing gradient accumulation due to lack of compute
      optimizer.step()
      optimizer.zero_grad()
      lr_scheduler.step()

      epoch_loss += losses.item()
      
  avg_loss = epoch_loss / len(train_loader)

  print(f"Average Loss: {avg_loss:.4f}")
  
  model.eval()
  metric.reset()
  
  with torch.no_grad():
      test_img, test_target = next(iter(test_loader))
      test_img = list(img.to(device) for img in test_img)
      
      predictions = model(test_img)
      print(f"\nPrediction Check:")
      print(f"  Num boxes predicted: {len(predictions[0]['boxes'])}")
      print(f"  Scores: {predictions[0]['scores'][:5]}")  # Top 5 scores
      print(f"  Labels: {predictions[0]['labels'][:5]}")  # Top 5 labels
      print(f"  Labels (csv): {predictions[0]['labels'][:5] - 1}")

      #visualize_predictions(test_img[0], test_target[0], predictions[0], score_threshold=0.3)

  with torch.no_grad(): # Por cada epoch pruebo el accuracy del modelo con el dataset de testing
    for test_inputs, test_targets in tqdm(test_loader, desc='Calculating mAP'):
      test_inputs = list(image.to(device) for image in test_inputs) 
      test_outputs = model(test_inputs)

      test_outputs_cpu = []
      for out in test_outputs:
          # Movemos cada tensor del diccionario a cpu
          test_outputs_cpu.append({k: v.cpu() for k, v in out.items()})
          
      test_targets_cpu = []
      for t in test_targets:
          # Los targets originales podrían estar en GPU o CPU, aseguramos CPU
          test_targets_cpu.append({k: v.cpu() for k, v in t.items()})

        # 3. Actualizar métrica con datos en RAM (CPU)
      #show_image_and_bounding(test_inputs[0], test_targets[0])
      metric.update(test_outputs_cpu, test_targets_cpu)

    result = metric.compute()
    print(f"Validation mAP (IoU=0.50:0.95): {result['map_50']:.4f}")

    if result['map_50'] > 0.8: 
        torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")

Epoch 1/100


  0%|          | 0/414 [00:00<?, ?it/s]c:\Users\srene\miniconda3\envs\pytdml\lib\site-packages\torch\nn\functional.py:3172: UserWarning: The operator 'aten::binary_cross_entropy' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch._C._nn.binary_cross_entropy(input, target, weight, reduction_enum)
100%|██████████| 414/414 [04:43<00:00,  1.46it/s]


Average Loss: 0.1351

Prediction Check:
  Num boxes predicted: 100
  Scores: tensor([0.3451, 0.3348, 0.3184, 0.3182, 0.3150], device='privateuseone:0')
  Labels: tensor([7, 7, 7, 4, 7], device='privateuseone:0')
  Labels (csv): tensor([6, 6, 6, 3, 6], device='privateuseone:0')


Calculating mAP: 100%|██████████| 66/66 [00:21<00:00,  3.01it/s]


Validation mAP (IoU=0.50:0.95): 0.0001
Epoch 2/100


...

Average Loss: 0.1305

Prediction Check:
  Num boxes predicted: 100
  Scores: tensor([0.3451, 0.3348, 0.3184, 0.3182, 0.3150], device='privateuseone:0')
  Labels: tensor([7, 7, 7, 4, 7], device='privateuseone:0')
  Labels (csv): tensor([6, 6, 6, 3, 6], device='privateuseone:0')


Calculating mAP: 100%|██████████| 66/66 [00:21<00:00,  3.05it/s]


Validation mAP (IoU=0.50:0.95): 0.0001
Epoch 6/100


 15%|█▍        | 62/414 [00:42<04:01,  1.46it/s]


KeyboardInterrupt: 

In [ ]:
# === SAM3 BACKBONE VISUALIZATION (STANDALONE) ===

import torch
import matplotlib.pyplot as plt
import numpy as np
import cv2
from transformers import AutoModel, AutoProcessor
from data.transforms import get_valid_transforms

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))

# 1. Load standalone SAM3 Model
print("Loading standalone SAM3 for visualization...")
try:
    # Attempt to load using AutoModel/AutoProcessor
    # Note: 'trust_remote_code=True' needed for custom models
    processor = AutoProcessor.from_pretrained("facebook/sam3", trust_remote_code=True)
    model_sam3 = AutoModel.from_pretrained("facebook/sam3", trust_remote_code=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_sam3.to(device)
    print("SAM3 loaded successfully.")
except Exception as e:
    print(f"Error loading SAM3: {e}")
    processor = None
    model_sam3 = None

if model_sam3:
    # 2. Get a Validation Image
    import pandas as pd
    import os
    from PIL import Image

    val_df = pd.read_csv(CSV_PATH_VAL)
    sample_row = val_df.iloc[0]
    image_id = sample_row['image_filename']
    image_path = os.path.join(VAL_PATH, image_id)
    
    print(f"Visualizing: {image_path}")
    raw_image = Image.open(image_path).convert("RGB")
    width_orig, height_orig = raw_image.size
    
    # Get Ground Truth Box for prompting SAM
    records = val_df[val_df['image_filename'] == image_id]
    input_boxes = []
    for _, row in records.iterrows():
        xc, yc, w, h = row['x'], row['y'], row['width'], row['height']
        x1, y1 = xc - w/2, yc - h/2
        x2, y2 = xc + w/2, yc + h/2
        input_boxes.append([x1, y1, x2, y2])
    
    import gc
    
    # A. Process Image 
    # We allow the processor to resize the image (usually to 1024x1024)
    inputs = processor(images=raw_image, return_tensors="pt").to(device)
    
    # B. Ensure metadata keys exist
    if 'reshaped_input_sizes' not in inputs:
        h, w = inputs['pixel_values'].shape[-2:]
        inputs['reshaped_input_sizes'] = torch.tensor([[h, w]], device=device)
    if 'original_sizes' not in inputs:
        inputs['original_sizes'] = torch.tensor([[height_orig, width_orig]], device=device)

    # C. Calculate Box Scaling
    reshaped_h, reshaped_w = inputs['reshaped_input_sizes'][0].tolist()
    scale_x = reshaped_w / width_orig
    scale_y = reshaped_h / height_orig
    
    # Check if image is massive (Debug)
    print(f"Original Size: {width_orig}x{height_orig}")
    print(f"Resized Input Size: {reshaped_w}x{reshaped_h}")
    
    # Scale boxes
    box_tensor = torch.tensor([input_boxes], device=device).float()
    scaled_boxes = box_tensor.clone()
    scaled_boxes[:, :, 0] *= scale_x
    scaled_boxes[:, :, 2] *= scale_x
    scaled_boxes[:, :, 1] *= scale_y
    scaled_boxes[:, :, 3] *= scale_y

    # D. Prepare Dummy Text Inputs
    if hasattr(processor, "tokenizer"):
        dummy_text_inputs = processor.tokenizer(
            [""], padding="max_length", max_length=2, truncation=True, return_tensors="pt"
        )
        inputs["input_ids"] = dummy_text_inputs["input_ids"].to(device)
        inputs["attention_mask"] = dummy_text_inputs["attention_mask"].to(device)
    else:
        inputs["input_ids"] = torch.zeros((1, 2), dtype=torch.long, device=device)
        inputs["attention_mask"] = torch.ones((1, 2), dtype=torch.long, device=device)

    # E. Run Inference (BATCH_SIZE = 1)
    inference_model = model_sam3
    if hasattr(model_sam3, 'detector_model'):
        inference_model = model_sam3.detector_model

    # --- ULTRA SAFE BATCHING ---
    BATCH_SIZE = 1  # Reduce to 1 to minimize RAM usage
    all_boxes = scaled_boxes[0] 
    total_boxes = len(all_boxes)
    
    print(f"Processing {total_boxes} boxes in batches of {BATCH_SIZE}...")
    
    accumulated_masks = []
    accumulated_scores = []
    
    import math
    num_batches = math.ceil(total_boxes / BATCH_SIZE)

    # Pre-extract shared inputs to avoid dictionary overhead
    pixel_values = inputs["pixel_values"]
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    for i in range(num_batches):
        start_idx = i * BATCH_SIZE
        end_idx = min((i + 1) * BATCH_SIZE, total_boxes)
        
        # 1. Create a minimal input dictionary for just this step
        chunk_boxes = all_boxes[start_idx:end_idx].unsqueeze(0)
        
        step_inputs = {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "input_boxes": chunk_boxes,
            # Pass original/reshaped sizes if model requires them in forward (SAM3 usually doesn't, but good to be safe)
        }
        
        # 2. Run Inference
        try:
            with torch.no_grad():
                outputs = inference_model(**step_inputs)
                
            # 3. Extract and DETACH data immediately
            # .clone() ensures we don't keep a ref to the graph
            mask_cpu = outputs.pred_masks.detach().cpu().clone()
            score_cpu = outputs.iou_scores.detach().cpu().clone()
            
            accumulated_masks.append(mask_cpu)
            accumulated_scores.append(score_cpu)
            
            # 4. Aggressive Cleanup
            del outputs
            del step_inputs
            del mask_cpu
            del score_cpu
            
        except RuntimeError as e:
            # Catch OOM specifically to warn user
            if "out of memory" in str(e):
                print(f"OOM at index {i}. Try restarting kernel.")
                break
            else:
                raise e

        # 5. Clear Caches
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect() # Force CPU RAM cleanup

    print("Aggregating results...")
    final_pred_masks = torch.cat(accumulated_masks, dim=1)
    final_iou_scores = torch.cat(accumulated_scores, dim=1)
    print("Inference complete.")

    # 4. Visualize
    # We use the concatenated results here
    masks = processor.image_processor.post_process_masks(
        final_pred_masks, 
        inputs["original_sizes"].cpu(), 
        inputs["reshaped_input_sizes"].cpu()
    )
    
    scores = final_iou_scores
    
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    
    # Loop through the original input_boxes list
    for i, box in enumerate(input_boxes):
        show_box(box, plt.gca())
        # Select best mask for this box
        best_mask_idx = scores[0, i].argmax()
        show_mask(masks[0][i, best_mask_idx].numpy(), plt.gca())
        
    plt.axis('off')
    plt.title(f"SAM3 Segmentation - {image_id}")
    plt.show()

Loading standalone SAM3 for visualization...


Loading weights: 100%|██████████| 1797/1797 [00:02<00:00, 715.51it/s, Materializing param=tracker_neck.fpn_layers.3.proj2.weight]                                                 


SAM3 loaded successfully.
Visualizing: ./RIVA/images/images/val\LSIL_29_1.png
Original Size: 1024x1024
Resized Input Size: 1008x1008
Processing 17 boxes in batches of 1...


In [ ]:
# === SAM3 BACKBONE TRAINING LOOP ===

# 1. IMPORTS
import torch
#import torch_directml
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Custom imports from our new implementation
try:
    from models.sam3_rcnn import get_sam3_faster_rcnn
    from data.transforms import get_train_transforms, get_valid_transforms
except ImportError as e:
    print(f"Import Error: {e}. Make sure 'models' and 'data' folders are in the path.")

# 2. DEVICE SETUP
try:
    device = torch_directml.device()
    print(f"Using DirectML device: {device}")
except:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

# 3. DATASETS & DATALOADERS
# Using the BethesdaDataset class defined in previous cells
# But applying the NEW transforms optimized for SAM3 (1008x1008)

print("Initializing Datasets with SAM3 transforms (1008x1008)...")
train_ds_sam3 = BethesdaDataset(
    csv_file=CSV_PATH_TRAIN, 
    root_dir=TRAIN_PATH, 
    transforms=get_train_transforms()
)
test_ds_sam3 = BethesdaDataset(
    csv_file=CSV_PATH_VAL, 
    root_dir=VAL_PATH, 
    transforms=get_valid_transforms()
)

def collate_fn_sam(batch):
    return tuple(zip(*batch))

# Determine batch size. SAM3 is heavy.BATCH_SIZE = 2

BATCH_SIZE = 2

train_loader_sam3 = DataLoader(
    train_ds_sam3, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn_sam
)
test_loader_sam3 = DataLoader(
    test_ds_sam3, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn_sam
)

# 4. MODEL INITIALIZATION
print("Loading FasterRCNN with SAM3 backbone...")
num_classes = 10 # 9 classes + background
# Using Hugging Face 'facebook/sam3' by default
model = get_sam3_faster_rcnn(num_classes=num_classes)
model.to(device)

# 5. OPTIMIZER
# Filter parameters requiring gradients (SAM3 backbone is frozen by default)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(params, lr=1e-3, weight_decay=1e-4)

# 6. TRAINING LOOP
num_epochs = 100

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
    
    # --- TRAINING ---
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader_sam3, desc=f"Training Epoch {epoch+1}")
    for i, (images, targets) in enumerate(pbar):
        # Move to device and ensure float tensors
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass returns dictionary of losses
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values()) # Sum all losses
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        pbar.set_postfix({'loss': f"{losses.item():.4f}"})
        
    avg_loss = total_loss / len(train_loader_sam3)
    print(f"Average Training Loss: {avg_loss:.4f}")
    
    # --- VALIDATION (mAP) ---
    print("Validating...")
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    
    with torch.no_grad():
        for images, targets in tqdm(test_loader_sam3, desc="Validation"):
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Forward pass in eval mode returns detections (list of dicts)
            outputs = model(images)
            
            # Send to metric. Both outputs and targets are lists of dicts.
            # Outputs on device, targets on device - metric handles it.
            # Some metrics need CPU conversion, but torchmetrics handles recent versions.
            # Ensuring CPU just in case for complex metrics if it fails.
            outputs_cpu = [{k: v.cpu() for k, v in t.items()} for t in outputs]
            targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
            metric.update(outputs_cpu, targets_cpu)
            
    results = metric.compute()
    # map is 0.50:0.95 (COCO standard)
    print(f"Validation Results:")
    print(f"mAP (0.50:0.95): {results['map']:.4f}")
    print(f"mAP (0.50): {results['map_50']:.4f}")
    print(f"mAP (0.75): {results['map_75']:.4f}")
    
    # Save Model
    # torch.save(model.state_dict(), f"checkpoints/riva_sam3_epoch_{epoch+1}.pth")


Using device: cpu
Initializing Datasets with SAM3 transforms (1008x1008)...
Loading FasterRCNN with SAM3 backbone...
Loading SAM3 model from facebook/sam3...


Loading weights: 100%|██████████| 1468/1468 [00:02<00:00, 635.96it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                       



--- Epoch 1/10 ---


Training Epoch 1:  10%|▉         | 40/414 [47:42<7:26:05, 71.57s/it, loss=0.9536] 


KeyboardInterrupt: 